In [9]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from src.preprocessing import get_preprocessor

In [18]:
train = pd.read_csv('../data/train.csv')

X = train.drop(columns='SalePrice')
y = train['SalePrice']

y_log = np.log1p(y)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    random_state=42,
    verbosity=-1
)

lgbm_pipeline = Pipeline([
    ('preprocessor', get_preprocessor(X)),
    (['model', model]),
])

scores = cross_val_score(
    estimator=lgbm_pipeline,
    X=X,
    y=y_log,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

rmse_scores = -scores

print("Fold RMSE:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

Fold RMSE: [0.13828203 0.11962653 0.16638473 0.13382441 0.11582946]
Mean RMSE: 0.13478943282142114
Std RMSE: 0.017892941158296025


In [ ]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    random_state=42,
    n_jobs=-1,
)

xgb_pipeline = Pipeline([
    ('preprocessor', get_preprocessor(X)),
    (['model', model]),
])

scores = cross_val_score(
    estimator=xgb_pipeline,
    X=X,
    y=y_log,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

rmse_scores = -scores

print("Fold RMSE:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

Fold RMSE: [0.1394501  0.11899522 0.17231646 0.13680268 0.11817054]
Mean RMSE: 0.13714699868701213
Std RMSE: 0.019656587404051358


In [24]:
def evaluate_catboost(X, y, cat_features, model_params=None, n_splits=5, seed=42, early_stopping_rounds=None):
    if model_params is None:
        model_params = {}
        
    cv = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )
    
    rmse_scores = []
    best_iterations = []
    
    for train_idx, val_idx in cv.split(X, y):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        
        preprocessor = get_preprocessor(X_train, model_type='catboost')
        
        X_train_processed = preprocessor.fit_transform(X_train, y_train)
        X_val_processed = preprocessor.transform(X_val)
        
        model = CatBoostRegressor(
            random_seed=seed,
            verbose=False,
            **model_params
        )
        
        model.fit(
            X_train_processed, 
            y_train,
            cat_features=cat_features,
            eval_set=(X_val_processed, y_val),
            early_stopping_rounds=early_stopping_rounds)
        
        best_iterations.append(model.get_best_iteration())
        
        preds = model.predict(X_val_processed)
        
        rmse = np.sqrt(np.mean((y_val.values - preds) ** 2))
        
        rmse_scores.append(rmse)
        
    rmse_scores = np.array(rmse_scores)

    return {
        "rmse_scores": rmse_scores,
        "mean_rmse": rmse_scores.mean(),
        "std_rmse": rmse_scores.std(),
        "best_iterations": best_iterations,
    }

In [22]:
catboost_preprocessor = get_preprocessor(X, model_type='catboost')

cat_features = X.select_dtypes(include=['object', 'string']).columns.tolist()

cat_features.append('MSSubClass')

n_cat_features = len(cat_features)

cat_features_indices = list(range(n_cat_features))

In [25]:
result = evaluate_catboost(
    X=X,
    y=y_log,
    cat_features=cat_features_indices
)

print(f"Mean RMSE: {result['mean_rmse']:.6f}")
print(f"Std RMSE:  {result['std_rmse']:.6f}")
print("Fold RMSE:", np.round(result["rmse_scores"], 6))
print()
print("Best iterations:", result["best_iterations"])

Mean RMSE: 0.125924
Std RMSE:  0.016147
Fold RMSE: [0.134946 0.112781 0.150808 0.125903 0.105181]

Best iterations: [921, 948, 894, 999, 973]
